# امبدینگ دیتاست RAG با bge-m3 روی Google Colab

این نوت‌بوک **فقط بخش امبدینگ** پروژهٔ شما را روی GPU کولب اجرا می‌کند و ایندکس نهایی FAISS را در گوگل‌درایو ذخیره می‌کند — همان خط لولهٔ `scripts/embed_dataset.py` با همان مدل، همان پارامترها و خروجی کاملاً سازگار با پروژه.

| مورد | مقدار (دقیقاً مثل docker-compose خودتان) |
|---|---|
| مدل | bge-m3 (GGUF نسخهٔ F16/FP16) |
| Pooling | `cls` |
| نرمال‌سازی | L2 (شباهت کسینوسی در FAISS) |
| چانکینگ | ۴ پاراگراف، هم‌پوشانی ۱، سقف ۳۰۰۰ کاراکتر، پاراگراف حداکثر ۱۵۰۰ |
| ایندکس | FAISS `IndexFlatIP` + `metadata.json` |
| ابعاد بردار | 1024 |

**راه‌اندازی — فقط ۳ قدم:**
1. منوی کولب: **Runtime → Change runtime type → T4 GPU** (تأیید اینکه GPU وصل شده)
2. سلول‌ها را از بالا به پایین اجرا کنید (Ctrl+F9 = Run all)
3. دیتاست: اگر `data.csv` را در ریشهٔ گوگل‌درایو بگذارید خودکار پیدا می‌شود؛ وگرنه سلول دوم از شما آپلود می‌گیرد

**زمان تقریبی (روی T4):**

| مرحله | زمان |
|---|---|
| نصب llama-cpp-python با CUDA | ۱ تا ۱۲ دقیقه (اول ویل پیش‌ساخته + پیش‌بارگذاری CUDA؛ اگر نشد کامپایل از سورس) |
| دانلود مدل (~1.2GB) | ۱ تا ۳ دقیقه |
| امبدینگ ~۱۴٫۷۰۰ متن یکتا (۱۵٫۲۰۰ چانک) | **حدود ۳۰ تا ۹۰ دقیقه** — به‌جای ۷ ساعتِ GTX 1650! |

**خروجی (Drive → پوشهٔ `RAG_Embed_Output`):**
- `index/index.faiss` و `index/metadata.json` ← محتویات پوشهٔ `index` را در `storage/index/` پروژهٔ خودتان کپی کنید
- `RAG_index.zip` ← همان دو فایل به‌صورت یک زیپ
- `embeddings_partial.npy` + `embeddings_index.json` ← کشِ ادامهٔ کار؛ اگر کولب وسط کار قطع شد کافی است دوباره Run all کنید و از همان‌جا ادامه می‌دهد (بعد از موفقیت خودکار پاک می‌شوند)

> **سازگاری بردارها:** چون مدل و pooling دقیقاً همان سرویس امبدینگ docker خودتان است (bge-m3-F16 + CLS)، بردارهای پرس‌وجوی بعدی اپلیکیشن با این ایندکس کاملاً هم‌خوان است — بدون هیچ تغییری در کد پروژه.

In [ ]:
# ۱) بررسی GPU — اگر خطا داد یا GPU ندیدید:
#    Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
import platform
print('Python:', platform.python_version())

In [ ]:
# ۲) اتصال گوگل‌درایو و پیدا کردن data.csv
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_OUT = '/content/drive/MyDrive/RAG_Embed_Output'   # پوشهٔ خروجی در گوگل‌درایو
INDEX_DIR = os.path.join(DRIVE_OUT, 'index')
os.makedirs(INDEX_DIR, exist_ok=True)

# اگر data.csv در درایو هست خودکار پیدا می‌شود (یا آدرس پوشهٔ خودتان را اضافه کنید)
DATA_CANDIDATES = [
    '/content/drive/MyDrive/data.csv',
    '/content/drive/MyDrive/RAG/data.csv',
    '/content/drive/MyDrive/project/data/data.csv',
    '/content/data.csv',
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    from google.colab import files as colab_files
    print('فایل data.csv (از پوشهٔ data پروژه) را آپلود کنید…')
    uploaded = colab_files.upload()
    assert uploaded, 'هیچ فایلی آپلود نشد!'
    src_name = next(iter(uploaded))
    DATA_PATH = '/content/data.csv'
    shutil.copy(src_name, DATA_PATH)

print('DATA_PATH =', DATA_PATH)
print('خروجی ایندکس →', DRIVE_OUT)

In [ ]:
# ۳) نصب llama-cpp-python با پشتیبانی CUDA + faiss
import glob
import importlib
import os
import site
import subprocess
import sys
from pathlib import Path


def sh(cmd: str) -> int:
    return subprocess.run(cmd, shell=True, check=False).returncode


def llama_import_ok() -> bool:
    # هر خطای ایمپورت را می‌گیرد (RuntimeError/OSError/…) نه فقط ImportError
    try:
        importlib.invalidate_caches()
        import llama_cpp  # noqa: F401
        return True
    except Exception:  # noqa: BLE001
        sys.modules.pop('llama_cpp', None)
        return False


def nvidia_lib_paths() -> list[str]:
    # همهٔ مسیرهای محتمل کتابخانه‌های CUDA روی کولب
    patterns: list[str] = []
    roots = list(site.getsitepackages()) + [str(site.getusersitepackages())]
    for root in roots:
        if root:
            patterns.append(root.rstrip('/') + '/nvidia/*/lib/lib*.so*')
    patterns += [
        '/usr/local/lib/python3*/dist-packages/nvidia/*/lib/lib*.so*',
        '/usr/lib/python3*/dist-packages/nvidia/*/lib/lib*.so*',
        '/usr/local/cuda*/lib64/libcudart.so*',
        '/usr/lib/x86_64-linux-gnu/libcudart.so*',
    ]
    paths: list[str] = []
    for pat in patterns:
        paths += glob.glob(pat)
    return sorted(set(paths))


def cudart_on_disk() -> bool:
    # آیا libcudart جایی روی سیستم/سایت‌پکیج‌ها موجود است؟
    return any(os.path.basename(p).startswith('libcudart')
               for p in nvidia_lib_paths())


def preload_nvidia_libs() -> list[str]:
    # کتابخانه‌های CUDAِ موجود (PyTorch کولب / سیستم) را با RTLD_GLOBAL
    # پیش‌بارگذاری می‌کند تا libllama.so بتواند آن‌ها را پیدا کند.
    import ctypes
    order = {'libcudart': 0, 'libcublasLt': 1, 'libcublas': 2}
    loaded: list[str] = []
    for path in sorted(nvidia_lib_paths(),
                       key=lambda p: next((r for k, r in order.items()
                                           if os.path.basename(p).startswith(k)), 99)):
        if not os.path.basename(path).startswith(tuple(order)):
            continue
        try:
            ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
            loaded.append(os.path.basename(path))
        except OSError:
            pass
    return loaded


def cuda_libs_present() -> bool:
    # آیا بیلد نصب‌شده واقعاً بک‌اند CUDA را دارد؟
    try:
        import llama_cpp
    except Exception:  # noqa: BLE001
        return False
    pkg = Path(llama_cpp.__file__).parent
    hits = glob.glob(str(pkg / '**' / '*ggml*cuda*'), recursive=True)
    hits += glob.glob(str(pkg / '**' / 'libcudart*'), recursive=True)
    return len(hits) > 0


print('نصب faiss و huggingface_hub …')
sh('pip install -q faiss-cpu huggingface_hub')

print('تلاش برای ویل پیش‌ساختهٔ CUDA (سریع، حدود ۱ دقیقه) …')
wheel_installed = sh('pip install -q llama-cpp-python '
                     '--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 '
                     '--only-binary=:all:') == 0

CUDA_OK = wheel_installed and llama_import_ok() and cuda_libs_present()

# ویل‌های پیش‌ساخته کتابخانه‌های runtime CUDA را همراه نمی‌آورند؛
# از همان کتابخانه‌هایی که PyTorch روی کولب آورده استفاده می‌کنیم.
if wheel_installed and not CUDA_OK:
    if not cudart_on_disk():
        print('runtime کتابخانه‌های CUDA پیدا نشد؛ نصب از pip …')
        sh('pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12')
    print('پیش‌بارگذاری کتابخانه‌های CUDA موجود …')
    loaded = preload_nvidia_libs()
    print('  پیش‌بارگذاری شد:', ', '.join(loaded[:5]), '…' if len(loaded) > 5 else '')
    CUDA_OK = llama_import_ok() and cuda_libs_present()
    if CUDA_OK:
        print('✓ ویل پیش‌ساخته با پیش‌بارگذاری کتابخانه‌ها فعال شد')

# آخرین راه مطمئن: کامپایل از سورس با CUDA خودِ ماشین (همیشه سازگار است)
if not CUDA_OK:
    print('کامپایل از سورس با CUDA (~۸ تا ۱۲ دقیقه) — لطفاً صبر کنید …')
    sh('pip uninstall -y -q llama-cpp-python')
    rc = sh('CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q '
            '--force-reinstall --no-cache-dir llama-cpp-python')
    CUDA_OK = (rc == 0) and llama_import_ok() and cuda_libs_present()
    assert CUDA_OK, ('نصب CUDA ممکن نشد! Runtime را روی T4 GPU بگذارید '
                     'و این سلول را دوباره اجرا کنید.')

import llama_cpp
print('نسخهٔ llama-cpp-python:', llama_cpp.__version__, '| بک‌اند CUDA: فعال ✓')

In [ ]:
# ۴) دانلود مدل bge-m3 نسخهٔ F16 (~1.2GB)
#    اگر خودتان فایل bge-m3-F16.gguf را در درایو گذاشته باشید از همان استفاده می‌شود
import os
from huggingface_hub import hf_hub_download

LOCAL_CANDIDATES = [
    '/content/drive/MyDrive/bge-m3-F16.gguf',
    '/content/drive/MyDrive/models/bge-m3-F16.gguf',
]
MODEL_PATH = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)

if MODEL_PATH:
    print('✓ استفاده از نسخهٔ موجود در درایو:', MODEL_PATH)
else:
    # دو مخزن معتبر؛ هرکدام که در دسترس بود دانلود می‌شود
    for repo, fname in [('gpustack/bge-m3-GGUF', 'bge-m3-FP16.gguf'),
                        ('CompendiumLabs/bge-m3-gguf', 'bge-m3-f16.gguf')]:
        try:
            print(f'دانلود {fname} از {repo} …')
            MODEL_PATH = hf_hub_download(repo_id=repo, filename=fname,
                                         local_dir='/content/models')
            break
        except Exception as exc:  # noqa: BLE001
            print('✗ ناموفق:', exc)
    assert MODEL_PATH, 'دانلود مدل ناموفق بود!'

print(MODEL_PATH, f'({os.path.getsize(MODEL_PATH) / 1e9:.2f} GB)')

## ماژول‌های هستهٔ پروژه

سلول بعدی چهار ماژول اصلی پروژه — `exceptions` ، `models` ، `chunker` و `vector_store` — را **عیناً و بدون هیچ تغییری** از `src/core` در کولب بازسازی می‌کند تا چانکینگ، متادیتا و ساختار ایندکس دقیقاً همان خروجی `scripts/embed_dataset.py` باشد.

In [ ]:
# ─── ماژول‌های هستهٔ پروژه — کپیِ عیناً از src/core (بدون هیچ تغییری) ───
# chunker.py   : CSVLoader + ParagraphChunker (همان چانکینگ اسکریپت اصلی)
# vector_store.py : VectorStore (همان ساخت و ذخیرهٔ ایندکس FAISS)
from pathlib import Path
import sys
sys.path.insert(0, '')

PKG = Path('ragpkg')
PKG.mkdir(exist_ok=True)
(PKG / '__init__.py').write_text('', encoding='utf-8')

(PKG / 'exceptions.py').write_text(r'''"""خطاهای سفارشی سیستم RAG — همه از یک پایه مشترک ارث می‌برند."""


class RAGSystemError(Exception):
    """پایهٔ تمام خطاهای پروژه."""


class ConfigurationError(RAGSystemError):
    """تنظیمات نامعتبر یا ناقص."""


class APIConnectionError(RAGSystemError):
    """عدم توانایی در اتصال به سرویس (مدل زبانی یا امبدینگ)."""


class APIResponseError(RAGSystemError):
    """پاسخ ناموفق یا نامعتبر از سرویس (کد 4xx/5xx یا ساختار خراب)."""


class DatasetError(RAGSystemError):
    """مشکل در خواندن یا اعتبارسنجی فایل دیتاست CSV."""


class IndexNotFoundError(RAGSystemError):
    """ایندکس FAISS هنوز ساخته یا بارگذاری نشده است."""
''', encoding='utf-8')
(PKG / 'models.py').write_text(r'''"""ساختارهای دادهٔ مشترک سیستم (dataclass ها)."""

from __future__ import annotations

from dataclasses import dataclass, field


@dataclass
class Document:
    """یک سند ورودی؛ معادل یک ردیف از دیتاست CSV (یک فایل PDF)."""

    source: str        # نام فایل/منبع (مقدار ستون filename)
    content: str       # متن کامل فایل (مقدار ستون content)


@dataclass
class Chunk:
    """یک قطعهٔ چانک‌شدهٔ آمادهٔ امبدینگ."""

    text: str                      # متن چانک
    source: str                    # منبع (نام فایل)
    chunk_index: int               # شمارهٔ چانک در همان سند
    paragraph_indices: tuple = ()  # شمارهٔ پاراگراف‌های پوشش‌داده‌شده
    char_count: int = 0            # طول متن چانک


@dataclass
class RetrievalResult:
    """نتیجهٔ یک جست‌وجوی معنایی (یک چانک بازیابی‌شده)."""

    text: str
    score: float        # شباهت کسینوسی (بالاتر = مرتبط‌تر)
    source: str
    chunk_index: int
    rank: int = 0       # رتبه در نتایج (از 1)


@dataclass
class ToolCall:
    """یک فراخوانی ابزار که مدل در خروجی خود اعلام کرده است."""

    name: str
    arguments: dict
    raw: str = ""       # بلوک خام متن فراخوانی


@dataclass
class ToolExecutionResult:
    """خروجی اجرای یک ابزار توسط ToolRegistry."""

    text: str                                   # متن نتیجه برای مدل
    sources: list = field(default_factory=list)  # منابع بازیابی‌شده (برای UI)


@dataclass
class PipelineResult:
    """نتیجهٔ نهایی یک اجرای کامل خط لولهٔ RAG."""

    answer: str = ""
    sources: list = field(default_factory=list)    # list[RetrievalResult]
    tool_calls: list = field(default_factory=list) # list[ToolCall]
    warnings: list = field(default_factory=list)   # list[str]
    error: str | None = None
    finished: bool = False

    def add_warning(self, message: str) -> None:
        """افزودن هشدار بدون تکرار."""
        if message not in self.warnings:
            self.warnings.append(message)
''', encoding='utf-8')
(PKG / 'chunker.py').write_text(r'''"""
چانکینگ پاراگراف‌محورِ بدونِ افتِ متن
====================================
ورودی هر سند (یک ردیف CSV = یک فایل PDF) به پاراگراف‌ها شکسته
می‌شود؛ سپس پاراگراف‌ها با روش «پنجرهٔ لغزان» در چانک‌هایی از چند
پاراگراف (+ هم‌پوشانی اختیاری) گروه می‌شوند.

تضمین‌ها:
  1) هیچ متنی از بین نمی‌رود — پس از چانک، پوشش کامل پاراگراف‌ها
     و برابری محتوایی (بدون فاصله‌ها) با متن اصلی راستی‌آزمایی می‌شود.
  2) پاراگراف‌های خیلی بلند (بلندتر از PARAGRAPH_MAX_CHARS) به‌صورت
     جمله‌به‌جمله و بدون افت به قطعات کوچک‌تر شکسته می‌شوند.
  3) طول هر چانک از CHUNK_MAX_CHARS عبور نمی‌کند (مگر یک پاراگراف
     تکیِ طولانی که خودش زیر سقف پاراگراف است).
"""

from __future__ import annotations
import io
import csv
import logging
import re
from pathlib import Path

from .exceptions import DatasetError
from .models import Chunk, Document
csv.field_size_limit(2**30)
logger = logging.getLogger("rag.chunker")

# جداسازی پاراگراف: یک خط خالی (فقط فاصله/تب)
_BLANK_LINE_RE = re.compile(r"\n\s*\n")
# جداسازی جمله: بعد از . ! ? … ؛ : یا هر خط جدیدی
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?…؛:])\s+|\n+")
# حذف همهٔ فاصله‌ها برای مقایسهٔ محتوایی
_WHITESPACE_RE = re.compile(r"\s+")


class CSVLoader:
    """خواندن دیتاست CSV با ستون‌های «نام فایل» و «محتوا».

    نام ستون‌های قابل قبول (به‌صورت case-insensitive):
      نام فایل:   filename | file_name | file | name | source | title
      محتوا:      content  | text | body | document | contents
    """

    NAME_COLUMNS = ("filename", "file_name", "file", "name", "source", "title")
    CONTENT_COLUMNS = ("content", "text", "body", "document", "contents")

    def __init__(self, name_column: str | None = None, content_column: str | None = None):
        self.name_column = name_column
        self.content_column = content_column

    # ── بارگذاری ─────────────────────────────────────────────
    def load(self, path: str | Path) -> list[Document]:
        """بارگذاری یک فایل CSV یا همهٔ فایل‌های CSV یک پوشه."""
        path = Path(path)
        if path.is_dir():
            files = sorted(path.glob("*.csv"))
        else:
            files = [path]
        if not files or not files[0].exists():
            raise DatasetError(f"مسیر دیتاست پیدا نشد: {path}")
        documents: list[Document] = []
        for file in files:
            documents.extend(self._load_single(file))
        if not documents:
            raise DatasetError("هیچ سند معتبری در دیتاست پیدا نشد (همه ردیف‌ها خالی‌اند؟).")
        return documents


    def _load_single(self, file: Path) -> list[Document]:
        documents: list[Document] = []

        # بایت‌های NUL (مثلاً از استخراج PDF) باعث «line contains NUL» می‌شوند؛
        # قبل از پارس CSV حذفشان می‌کنیم.
        raw = file.read_bytes()
        nul_count = raw.count(b"\x00")
        if nul_count:
            logger.warning("فایل «%s» شامل %d بایت NUL بود و پاک‌سازی شد.",
                           file.name, nul_count)
            raw = raw.replace(b"\x00", b"")
        text = raw.decode("utf-8-sig", errors="replace")

        with io.StringIO(text, newline="") as handle:
            reader = csv.DictReader(handle)
            if not reader.fieldnames:
                raise DatasetError(f"فایل «{file.name}» سطر هدر ندارد.")
            name_col = self._resolve_column(reader.fieldnames, self.NAME_COLUMNS,
                                            self.name_column, "نام فایل")
            content_col = self._resolve_column(reader.fieldnames, self.CONTENT_COLUMNS,
                                               self.content_column, "محتوا")
            for row_number, row in enumerate(reader, start=2):
                source = (row.get(name_col) or "").strip() or f"{file.stem}#row{row_number}"
                content = (row.get(content_col) or "").strip()
                if not content:
                    logger.warning("ردیف %d از «%s» محتوای خالی دارد و نادیده گرفته شد.",
                                   row_number, file.name)
                    continue
                documents.append(Document(source=source, content=content))
        logger.info("فایل «%s» خوانده شد: %d سند.", file.name, len(documents))
        return documents

    @staticmethod
    def _resolve_column(fieldnames, candidates, override, label) -> str:
        """پیدا کردن نام واقعی ستون بین نام‌های مجاز (یا ستون صریح کاربر)."""
        normalized = {name.strip().lower(): name for name in fieldnames if name}
        if override:
            if override.lower() in normalized:
                return normalized[override.lower()]
            raise DatasetError(f"ستون {label} با نام «{override}» در فایل پیدا نشد.")
        for candidate in candidates:
            if candidate in normalized:
                return normalized[candidate]
        raise DatasetError(
            f"ستون {label} پیدا نشد. نام‌های قابل قبول: {list(candidates)} — "
            f"ستون‌های موجود فایل: {list(fieldnames)}"
        )


class ParagraphChunker:
    """چانکر پنجره‌ای چندپاراگرافی با هم‌پوشانی؛ تضمین پوشش کامل متن."""

    def __init__(self, paragraphs_per_chunk: int = 4, overlap_paragraphs: int = 1,
                 max_chunk_chars: int = 3000, paragraph_max_chars: int = 1500):
        self.paragraphs_per_chunk = max(1, paragraphs_per_chunk)
        self.overlap_paragraphs = max(
            0, min(overlap_paragraphs, self.paragraphs_per_chunk - 1)
        )
        self.max_chunk_chars = max(200, max_chunk_chars)
        self.paragraph_max_chars = max(100, paragraph_max_chars)

    # ── مرحلهٔ ۱: شکستن به پاراگراف ──────────────────────────
    def split_paragraphs(self, text: str) -> list[str]:
        """شکستن متن به پاراگراف‌ها (با خط خالی) و ریزکردن پاراگراف‌های بلند."""
        paragraphs = [part.strip() for part in _BLANK_LINE_RE.split(text) if part.strip()]
        pieces: list[str] = []
        for paragraph in paragraphs:
            if len(paragraph) <= self.paragraph_max_chars:
                pieces.append(paragraph)
            else:
                pieces.extend(self._split_long_paragraph(paragraph))
        return pieces

    def _split_long_paragraph(self, paragraph: str) -> list[str]:
        """ریزکردن پاراگراف بلند به قطعات حداکثر paragraph_max_chars — بدون افت."""
        sentences = [s for s in _SENTENCE_SPLIT_RE.split(paragraph) if s]
        pieces: list[str] = []
        buffer = ""
        for sentence in sentences:
            # جملهٔ خودش از سقف بلندتر است → برش ناچارِ کاراکتری
            while len(sentence) > self.paragraph_max_chars:
                if buffer:
                    pieces.append(buffer)
                    buffer = ""
                pieces.append(sentence[: self.paragraph_max_chars])
                sentence = sentence[self.paragraph_max_chars:]
            if not sentence:
                continue
            if buffer and len(buffer) + len(sentence) + 1 > self.paragraph_max_chars:
                pieces.append(buffer)
                buffer = sentence
            else:
                buffer = f"{buffer} {sentence}".strip()
        if buffer:
            pieces.append(buffer)
        return pieces

    # ── مرحلهٔ ۲: پنجرهٔ لغزان روی پاراگراف‌ها ────────────────
    def chunk(self, document: Document) -> list[Chunk]:
        """تقسیم یک سند به چانک‌های چندپاراگرافی + راستی‌آزمایی پوشش."""
        pieces = self.split_paragraphs(document.content)
        if not pieces:
            return []
        chunks: list[Chunk] = []
        index = 0
        total = len(pieces)
        while index < total:
            # انتخاب پنجره: تا سقف تعداد پاراگراف و سقف کاراکتر
            window: list[int] = []
            char_count = 0
            for cursor in range(index, min(index + self.paragraphs_per_chunk, total)):
                piece = pieces[cursor]
                if window and char_count + len(piece) > self.max_chunk_chars:
                    break
                window.append(cursor)
                char_count += len(piece) + 2   # +2 برای جداساز \n\n
            text = "\n\n".join(pieces[i] for i in window)
            chunks.append(Chunk(
                text=text,
                source=document.source,
                chunk_index=len(chunks),
                paragraph_indices=tuple(window),
                char_count=len(text),
            ))
            # گام حرکت پنجره با احتساب هم‌پوشانی
            index += max(1, len(window) - self.overlap_paragraphs)
        self._verify(document, pieces, chunks)
        return chunks

    def chunk_all(self, documents: list[Document]) -> list[Chunk]:
        """چانک کردن همهٔ اسناد (شمارهٔ هر چانک نسبت به سند خودش است)."""
        all_chunks: list[Chunk] = []
        for document in documents:
            all_chunks.extend(self.chunk(document))
        return all_chunks

    # ── راستی‌آزمایی «بدون افت» ──────────────────────────────
    @staticmethod
    def _verify(document: Document, pieces: list[str], chunks: list[Chunk]) -> None:
        """پوشش همهٔ پاراگراف‌ها توسط چانک‌ها + برابری محتوایی با متن اصلی."""
        covered: set[int] = set()
        for chunk in chunks:
            covered.update(chunk.paragraph_indices)
        if covered != set(range(len(pieces))):
            raise DatasetError(
                f"خطای پوشش چانک در سند «{document.source}»: "
                f"پاراگراف‌هایی از چانک‌ها جا مانده‌اند."
            )
        original = _WHITESPACE_RE.sub("", document.content)
        rebuilt = _WHITESPACE_RE.sub("", "".join(pieces))
        if original != rebuilt:
            raise DatasetError(
                f"افت متن در شکستن پاراگراف‌های سند «{document.source}» تشخیص داده شد."
            )

    # ── آمار ─────────────────────────────────────────────────
    @staticmethod
    def build_report(documents: list[Document], chunks: list[Chunk]) -> dict:
        """گزارش آماری چانکینگ برای نمایش در اسکریپت embed."""
        original_chars = sum(len(doc.content) for doc in documents)
        chunk_chars = [chunk.char_count for chunk in chunks]
        return {
            "documents": len(documents),
            "chunks": len(chunks),
            "original_chars": original_chars,
            "chunked_chars": sum(chunk_chars),
            "avg_chunk_chars": int(sum(chunk_chars) / len(chunk_chars)) if chunk_chars else 0,
            "max_chunk_chars": max(chunk_chars) if chunk_chars else 0,
            "sources": sorted({doc.source for doc in documents}),
        }
''', encoding='utf-8')
(PKG / 'vector_store.py').write_text(r'''"""
ذخیره‌گاه برداری FAISS
======================
مدیریت ایندکس ‎IndexFlatIP‎ (جست‌وجوی دقیق) روی بردارهای نرمال‌شده؛
بنابراین امتیازها معادل شباهت کسینوسی هستند. متن و متادیتای هر
چانک در فایل ‎metadata.json‎ کنار ‎index.faiss‎ نگهداری می‌شود.
"""

from __future__ import annotations

import json
import logging
import time
from pathlib import Path

import faiss
import numpy as np

from .exceptions import IndexNotFoundError
from .models import Chunk, RetrievalResult

logger = logging.getLogger("rag.vector_store")


class VectorStore:
    """ساخت، ذخیره، بارگذاری و جست‌وجوی ایندکس FAISS."""

    INDEX_FILE = "index.faiss"
    META_FILE = "metadata.json"

    def __init__(self, index_dir: str | Path):
        self.index_dir = Path(index_dir)
        self.index = None            # faiss.IndexFlatIP
        self.metadatas: list[dict] = []
        self.info: dict = {}

    # ── وضعیت ────────────────────────────────────────────────
    def is_ready(self) -> bool:
        """آیا ایندکس بارگذاری شده و حداقل یک بردار دارد؟"""
        return self.index is not None and self.index.ntotal > 0

    def try_load(self) -> bool:
        """بارگذاری ایندکس از دیسک در صورت وجود؛ در غیر این صورت False."""
        index_path = self.index_dir / self.INDEX_FILE
        meta_path = self.index_dir / self.META_FILE
        if not index_path.is_file() or not meta_path.is_file():
            return False
        try:
            self.index = faiss.read_index(str(index_path))
            payload = json.loads(meta_path.read_text(encoding="utf-8"))
            self.metadatas = payload.get("metadatas", [])
            self.info = payload.get("info", {})
            if len(self.metadatas) != self.index.ntotal:
                raise IndexNotFoundError(
                    "تعداد متادیتاها با تعداد بردارهای ایندکس هم‌خوانی ندارد؛ "
                    "ایندکس را دوباره بسازید."
                )
            logger.info("ایندکس بارگذاری شد: %d چانک از «%s».",
                        self.index.ntotal, self.index_dir)
            return True
        except (OSError, RuntimeError, json.JSONDecodeError, ValueError,
                IndexNotFoundError) as exc:
            self.index = None
            self.metadatas = []
            self.info = {}
            logger.error("بارگذاری ایندکس ناموفق بود: %s", exc)
            return False

    # ── ساخت و ذخیره ─────────────────────────────────────────
    def build(self, chunks: list[Chunk], embeddings: np.ndarray) -> None:
        """ساخت ایندکس از چانک‌ها و بردارهای آن‌ها."""
        if not chunks:
            raise IndexNotFoundError("چانکی برای ساخت ایندکس وجود ندارد.")
        embeddings = np.asarray(embeddings, dtype="float32")
        if embeddings.ndim != 2 or embeddings.shape[0] != len(chunks):
            raise ValueError("تعداد چانک‌ها با ابعاد ماتریس بردارها هم‌خوانی ندارد.")
        vectors = np.ascontiguousarray(embeddings)
        dimension = vectors.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(vectors)
        self.index = index
        self.metadatas = [
            {
                "source": chunk.source,
                "chunk_index": chunk.chunk_index,
                "text": chunk.text,
                "char_count": chunk.char_count,
            }
            for chunk in chunks
        ]
        self.info = {
            "dimension": dimension,
            "count": len(chunks),
            "sources": sorted({chunk.source for chunk in chunks}),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    def save(self) -> Path:
        """ذخیرهٔ ایندکس و متادیتا روی دیسک."""
        if self.index is None:
            raise IndexNotFoundError("ایندکسی برای ذخیره ساخته نشده است.")
        self.index_dir.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(self.index_dir / self.INDEX_FILE))
        payload = {"info": self.info, "metadatas": self.metadatas}
        (self.index_dir / self.META_FILE).write_text(
            json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8"
        )
        logger.info("ایندکس در «%s» ذخیره شد (%d چانک).", self.index_dir, self.index.ntotal)
        return self.index_dir

    # ── جست‌وجو ───────────────────────────────────────────────
    def search(self, query_vector: np.ndarray, top_k: int) -> list[RetrievalResult]:
        """جست‌وجوی بردار پرس‌وجو و برگرداندن مرتبط‌ترین چانک‌ها."""
        if not self.is_ready():
            return []
        vector = np.ascontiguousarray(
            query_vector.reshape(1, -1), dtype="float32"
        )
        if vector.shape[1] != self.index.d:
            raise IndexNotFoundError(
                f"ابعاد بردار پرس‌وجو ({vector.shape[1]}) با ابعاد ایندکس "
                f"({self.index.d}) فرق دارد؛ ایندکس را دوباره بسازید."
            )
        k = min(max(1, top_k), self.index.ntotal)
        scores, ids = self.index.search(vector, k)
        results: list[RetrievalResult] = []
        for rank, (score, idx) in enumerate(zip(scores[0].tolist(), ids[0].tolist())):
            if idx < 0:
                continue   # FAISS برای k بیش از ntotal مقدار -1 برمی‌گرداند
            meta = self.metadatas[idx]
            results.append(RetrievalResult(
                text=meta["text"],
                score=float(score),
                source=meta["source"],
                chunk_index=meta["chunk_index"],
                rank=rank + 1,
            ))
        return results

    def stats(self) -> dict:
        """آمار ایندکس برای نمایش در UI و ابزار get_knowledge_base_stats."""
        ready = self.is_ready()
        return {
            "ready": ready,
            "count": int(self.index.ntotal) if self.index is not None else 0,
            "dimension": int(self.index.d) if self.index is not None else 0,
            "sources": self.info.get("sources", []),
            "created_at": self.info.get("created_at", ""),
        }
''', encoding='utf-8')

for p in sorted(PKG.glob('*.py')):
    print(f'  ✓ {p}  ({len(p.read_text(encoding="utf-8").splitlines())} خط)')
print('ماژول‌های پروژه آماده شدند.')

In [ ]:
# ۵) خواندن دیتاست + چانکینگ (همان پارامترهای .env پروژه)
import sys
sys.path.insert(0, '')
from ragpkg.chunker import CSVLoader, ParagraphChunker

CHUNK_PARAGRAPHS = 4       # CHUNK_PARAGRAPHS
CHUNK_OVERLAP = 1          # CHUNK_OVERLAP_PARAGRAPHS
CHUNK_MAX_CHARS = 3000     # CHUNK_MAX_CHARS
PARAGRAPH_MAX_CHARS = 1500 # PARAGRAPH_MAX_CHARS

print('[1/4] خواندن دیتاست …')
documents = CSVLoader().load(DATA_PATH)
assert documents, 'دیتاست خالی است!'

print('[2/4] چانکینگ پاراگراف‌محور …')
chunker = ParagraphChunker(
    paragraphs_per_chunk=CHUNK_PARAGRAPHS,
    overlap_paragraphs=CHUNK_OVERLAP,
    max_chunk_chars=CHUNK_MAX_CHARS,
    paragraph_max_chars=PARAGRAPH_MAX_CHARS,
)
chunks = chunker.chunk_all(documents)
report = ParagraphChunker.build_report(documents, chunks)

line = '─' * 52
print(f'\n{line}\nگزارش چانکینگ (پاراگراف‌محور، بدون افت متن)\n{line}')
print(f"تعداد اسناد (فایل‌ها)      : {report['documents']}")
print(f"تعداد چانک‌ها              : {report['chunks']}")
print(f"مجموع متن اصلی             : {report['original_chars']:,} کاراکتر")
print(f"میانگین طول چانک           : {report['avg_chunk_chars']:,} کاراکتر")
print(f"بلندترین چانک              : {report['max_chunk_chars']:,} کاراکتر")
print(f'پوشش کامل متن              : تایید شد (بدون از دست رفتن هیچ متنی)')
print(line)

# حذف چانک‌های خالی (مثل اسکریپت اصلی)
non_empty = [c for c in chunks if c.text and c.text.strip()]
if len(non_empty) != len(chunks):
    print(f'⚠ {len(chunks) - len(non_empty)} چانک خالی حذف شد.')
chunks = non_empty

# حذف متن‌های تکراری: هر متن فقط یک‌بار امبد می‌شود (مثل اسکریپت اصلی)
unique_texts, seen, inverse = [], {}, []
for c in chunks:
    idx = seen.get(c.text)
    if idx is None:
        idx = len(unique_texts)
        seen[c.text] = idx
        unique_texts.append(c.text)
    inverse.append(idx)

print(f"چانک‌ها: {len(chunks):,} | یکتا: {len(unique_texts):,} | "
      f"تکراری (امبد نمی‌شود): {len(chunks) - len(unique_texts):,}")

In [ ]:
# ۶) بارگذاری مدل bge-m3 روی GPU — دقیقاً با تنظیمات docker شما
#    (-c 8192  -b 2048  -ub 2048  --pooling cls  -fa on  -ngl 99)
import time
import numpy as np
import llama_cpp
from llama_cpp import Llama

kwargs = dict(
    model_path=MODEL_PATH,
    embedding=True,
    n_gpu_layers=-1,                                      # -ngl 99
    n_ctx=8192,                                           # -c 8192
    n_batch=2048,                                         # -b 2048
    n_ubatch=2048,                                        # -ub 2048
    pooling_type=llama_cpp.LLAMA_POOLING_TYPE_CLS,        # --pooling cls
    verbose=False,
)
try:
    model = Llama(flash_attn=True, **kwargs)              # -fa on
except TypeError:  # نسخه‌های قدیمی‌تر flash_attn ندارند
    model = Llama(**kwargs)


def embed_texts(texts: list[str]) -> np.ndarray:
    """امبدینگ لیست متن‌ها — مثل EmbeddingClient پروژه (مرتب بر اساس index)."""
    if not texts:
        return np.zeros((0, 1), dtype='float32')
    resp = model.create_embedding(list(texts))
    data = sorted(resp['data'], key=lambda x: x.get('index', 0))
    return np.asarray([d['embedding'] for d in data], dtype=np.float32)


def normalize(matrix: np.ndarray) -> np.ndarray:
    """نرمال‌سازی L2 — همان EmbeddingService._normalize پروژه."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (matrix / norms).astype('float32')


print('[3/4] تست سلامت سرویس امبدینگ …')
t0 = time.time()
probe = embed_texts(['تست سلامت سرویس'])
DIM = int(probe.shape[1])
print(f'✓ سرویس آماده است | بُعد بردار: {DIM} | زمان تست: {time.time() - t0:.2f}s')

In [ ]:
# ۷) امبدینگ همهٔ متن‌ها با نوار پیشرفت + کشِ خودکار در گوگل‌درایو
#    اگر Runtime قطع شود: دوباره Run all کنید — از همان‌جا ادامه می‌دهد.
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

BATCH_SIZE = 32          # تعداد متن در هر فراخوانی
CHECKPOINT_EVERY = 25    # هر ۲۵ دسته، یک‌بار در درایو ذخیره کن
EMB_FILE = os.path.join(DRIVE_OUT, 'embeddings_partial.npy')
IDX_FILE = os.path.join(DRIVE_OUT, 'embeddings_index.json')

hashes = [hashlib.sha1(t.encode('utf-8')).hexdigest() for t in unique_texts]

done: dict[str, np.ndarray] = {}
if os.path.exists(EMB_FILE) and os.path.exists(IDX_FILE):
    prev = np.load(EMB_FILE)
    order = json.loads(Path(IDX_FILE).read_text(encoding='utf-8'))
    if len(order) == len(prev):
        done = dict(zip(order, prev))
    print(f'↻ ادامهٔ کار: {len(done):,} متن از قبل امبد شده (از کش درایو)')

todo = [i for i, h in enumerate(hashes) if h not in done]
print(f'امبدینگ {len(todo):,} متن در دسته‌های {BATCH_SIZE}تایی …')
pbar = tqdm(total=len(todo), unit='متن', dynamic_ncols=True)
t_start = time.time()
for bi, start in enumerate(range(0, len(todo), BATCH_SIZE)):
    idxs = todo[start:start + BATCH_SIZE]
    vecs = normalize(embed_texts([unique_texts[i] for i in idxs]))
    if vecs.shape[0] != len(idxs):
        raise RuntimeError('تعداد بردار دریافتی با تعداد متن برابر نیست!')
    for i, v in zip(idxs, vecs):
        done[hashes[i]] = v
    if (bi + 1) % CHECKPOINT_EVERY == 0 or start + BATCH_SIZE >= len(todo):
        order = [h for h in hashes if h in done]
        np.save(EMB_FILE, np.stack([done[h] for h in order]).astype(np.float32))
        Path(IDX_FILE).write_text(json.dumps(order), encoding='utf-8')
    pbar.update(len(idxs))
pbar.close()

unique_emb = np.stack([done[h] for h in hashes]).astype(np.float32)
assert unique_emb.shape[0] == len(unique_texts)
assert unique_emb.shape[1] == DIM
assert np.isfinite(unique_emb).all(), 'بردار نامعتبر (NaN/Inf) از مدل برگشت!'
dt = time.time() - t_start
print(f'✓ ماتریس بردارها: {unique_emb.shape} | زمان: {dt / 60:.1f} دقیقه | '
      f'{len(todo) / max(dt, 1):.1f} متن/ثانیه')

In [ ]:
# ۸) ساخت و ذخیرهٔ ایندکس FAISS در گوگل‌درایو + ساخت زیپ
import os
import shutil
import sys
from pathlib import Path

import numpy as np
sys.path.insert(0, '')
from ragpkg.vector_store import VectorStore

print('[4/4] ساخت و ذخیرهٔ ایندکس FAISS …')

# بازکردن بردارهای یکتا به ازای همهٔ چانک‌ها (مثل اسکریپت اصلی)
embeddings = unique_emb[np.asarray(inverse)]
assert embeddings.shape[0] == len(chunks), 'ناسازگاری تعداد بردار و چانک!'
assert np.isfinite(embeddings).all(), 'بردارهای نهایی نامعتبرند!'

vector_store = VectorStore(INDEX_DIR)
vector_store.build(chunks, embeddings.astype(np.float32))
vector_store.save()

info = vector_store.info
print(f"✓ ایندکس ذخیره شد: {INDEX_DIR}")
print(f"  چانک‌ها     : {info['count']:,}")
print(f"  ابعاد بردار : {info['dimension']}")
print(f"  منابع       : {len(info['sources'])} فایل")

# کش امبدینگ دیگر لازم نیست — پاک‌سازی (مثل اسکریپت اصلی)
for f in (EMB_FILE, IDX_FILE):
    try:
        os.remove(f)
    except OSError:
        pass

# زیپ یکپارچه برای دانلود راحت از درایو
zip_path = shutil.make_archive(os.path.join(DRIVE_OUT, 'RAG_index'),
                               'zip', root_dir=DRIVE_OUT, base_dir='index')
print('✓ زیپ آماده شد:', zip_path)

In [ ]:
# ۹) راستی‌آزمایی: بارگذاری مجدد ایندکس از دیسک + یک جست‌وجوی نمونه
import sys
import numpy as np
sys.path.insert(0, '')
from ragpkg.vector_store import VectorStore

vs_check = VectorStore(INDEX_DIR)
assert vs_check.try_load(), 'بارگذاری مجدد ایندکس ناموفق بود!'
print(f'✓ ایندکس از دیسک بارگذاری شد: {vs_check.index.ntotal:,} بردار | بُعد {vs_check.index.d}')

# یک پرس‌وجوی نمونه — پاسخ باید از مقالات مرتبط باشد
query = 'MQTT protocol security for IoT devices'
q_vec = normalize(embed_texts([query]))[0].astype(np.float32)
for r in vs_check.search(q_vec, 3):
    print(f'{r.rank}. score={r.score:.4f} | {r.source}')
    print('   ' + r.text[:90].replace('\n', ' ') + '…')

In [ ]:
# ۱۰) پایان! فایل‌های نهایی در گوگل‌درایو شما هستند
from pathlib import Path


def fmt_size(p: Path) -> str:
    s = p.stat().st_size
    return f'{s / 1e6:.1f} MB' if s < 1e9 else f'{s / 1e9:.2f} GB'


print('=' * 62)
print('✅ همه چیز کامل شد! فایل‌های نهایی در گوگل‌درایو:')
print('=' * 62)
for rel in ['index/index.faiss', 'index/metadata.json', 'RAG_index.zip']:
    p = Path(DRIVE_OUT) / rel
    print(f'  {rel:24s} {fmt_size(p) if p.exists() else "—"}')
print('=' * 62)
print('» روی سیستم خودتان:')
print('  1) از درایو، پوشهٔ RAG_Embed_Output را بگیرید')
print('  2) محتوای پوشهٔ index (index.faiss و metadata.json) را در')
print('     <پروژه>/storage/index/ کپی کنید (جای فایل‌های قبلی)')
print('  3) docker-compose را بالا بیاورید (سرویس امبدینگ برای پرس‌وجو لازم است)')
print('  4) python run_app.py — بدون هیچ تغییری در کد پروژه')
print()
print('نکته: مدل و pooling این ایندکس (bge-m3 + CLS) دقیقاً همان سرویس')
print('docker شماست؛ پرس‌وجوها بدون مشکل کار می‌کنند.')